# Fidelity quantum kernel

Build a small feature-map kernel matrix from pairwise state fidelities.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

A fidelity quantum kernel compares data-encoding states through squared overlaps.

In [2]:
data = np.asarray([[0.1, 0.2, -0.1], [0.7, -0.4, 0.3], [-0.5, 0.6, 0.8], [0.2, 0.9, -0.7]])

def feature_map(values):
    circuit = QuantumCircuit(3)
    for wire, value in enumerate(values):
        circuit.h(wire)
        circuit.rz(float(value), wire)
    for wire in range(2):
        circuit.rzz(float(values[wire] * values[wire + 1]), wire, wire + 1)
    return circuit

circuits = [feature_map(row) for row in data]

def kernel(states):
    return np.asarray([[abs(np.vdot(left, right)) ** 2 for right in states] for left in states])

def reference_kernel():
    return kernel([np.asarray(Statevector.from_instruction(c).data) for c in circuits])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_kernel)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]

def mettleq_kernel():
    states = [np.asarray(backend.run(c, shots=1, return_statevector=True).result().data(0)["statevector"]) for c in compiled]
    return kernel(states)

candidate, mettleq_ms, _ = benchmark(mettleq_kernel)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

Every entry of the symmetric kernel matrix is compared numerically.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/13_quantum_kernel.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="fidelity kernel matrix atol=4e-6",
    passed=error <= 4e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_kernel_error": error, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — fidelity kernel matrix atol=4e-6
SDK reference median: 0.871 ms
MettleQ median:       3.073 ms
Timing interpretation: the SDK reference was 3.528x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "fidelity kernel matrix atol=4e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"max_kernel_error": 3.8624053866342933e-07, "mettleq": [[0.9999996423721313, 0.7812464237213135, 0.6591801643371582, 0.7277976274490356], [0.7812464237213135, 0.9999997615814209, 0.45091772079467773, 0.4014614224433899], [0.6591801643371582, 0.45091772079467773, 0.9999996423721313, 0.3512904644012451], [0.7277976274490356, 0.4014614224433899, 0.3512904644012451, 0.9999997615814209]], "reference": [[0.9999999999999991, 0.7812466110760015, 0.65918055057

## What should you conclude?

Kernel workloads offer reuse and batching opportunities, but this small matrix is intended to show the contract clearly.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.